# 01 — Data Loading & Initial Inspection

**PACE Phase**: Plan
**Objective**: load the two LAPD datasets (2010–2019 and 2020–2024), verify their structure, align the columns, concatenate them, and produce an initial inspection that guides the decisions of the Analyze phase.

**Output**: `data/processed/crimes_merged.parquet` — combined dataset, not yet cleaned, with consistent data types.

## 1. Setup

In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_rows',None)          # Show all rows

pd.set_option('display.max_columns',None)       # Show all columns

pd.set_option('display.max_info_columns',200)   # df.info() shows 200 columns
                                                # (200 is an indicative number, used to show all columns;
                                                #  if there were more than 200 columns, a higher number would need to be set)

## 2. Loading the raw datasets

In [3]:
df1 = pd.read_csv('../../data/raw/crime-2010-2019.csv') # Read the DataFrame for 2010-2019 crimes
df2 = pd.read_csv('../../data/raw/crime-2020-2024.csv') # Read the DataFrame for 2020-2024 crimes

## 3. Column verification and alignment

Let's compare the column names of the two DataFrames before concatenation.

In [4]:
var1=df1.columns # Create the list of columns for DataFrames 'df1' and 'df2'
var2=df2.columns # to check whether the column names are the same for both DataFrames before the merge

print( f'The columns of the first df are: {var1}')
print( f'The columns of the second df are: {var2}')

The columns of the first df are: Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA ', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT',
       'LON'],
      dtype='object')
The columns of the second df are: Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT',
       'LON'],
      dtype='object')


In [5]:
var1==var2  # Compare the column names

array([ True,  True,  True,  True, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

### Issue detected

The comparison reveals a difference: in the 2010–2019 dataset the `AREA` column has a trailing space (`'AREA '`), while in the 2020–2024 dataset it doesn't.

**Solution**: apply `.str.strip()` to the column names of both DataFrames to normalize any spaces and ensure alignment.

In [6]:
df1.columns = df1.columns.str.strip() # Apply the '.strip' function to df1's column names to remove any blank spaces
df2.columns = df2.columns.str.strip() # Apply the '.strip' function to df2's column names to remove any blank spaces

In [7]:
new_col1 = df1.columns # Create the variable 'new_col1' containing a list of the column names resulting from applying
                       # the '.strip' function

new_col2 = df2.columns # Create the variable 'new_col2' containing a list of the column names resulting from applying
                       # the '.strip' function

new_col1==new_col2     # Compare the two lists just created as a further check that the '.strip' function fixed
                       # the errors found

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

## 4. Concatenating the two datasets

In [8]:
df = pd.concat([df1, df2], ignore_index=True) # pd.concat merges the two DataFrames 'df1' and 'df2'. Since the
                                              # 'axis' parameter is not specified, the default value (axis=0)
                                              # is used for a vertical concatenation. 'ignore_index=True' resets
                                              # the index progressively (0, 1, 2, ...).

print(df.shape)  # '.shape' displays the number of rows and columns

print(df.info()) # '.info' displays the DataFrame's information

(3138031, 28)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3138031 entries, 0 to 3138030
Data columns (total 28 columns):
 #   Column          Dtype  
---  ------          -----  
 0   DR_NO           int64  
 1   Date Rptd       object 
 2   DATE OCC        object 
 3   TIME OCC        int64  
 4   AREA            int64  
 5   AREA NAME       object 
 6   Rpt Dist No     int64  
 7   Part 1-2        int64  
 8   Crm Cd          int64  
 9   Crm Cd Desc     object 
 10  Mocodes         object 
 11  Vict Age        int64  
 12  Vict Sex        object 
 13  Vict Descent    object 
 14  Premis Cd       float64
 15  Premis Desc     object 
 16  Weapon Used Cd  float64
 17  Weapon Desc     object 
 18  Status          object 
 19  Status Desc     object 
 20  Crm Cd 1        float64
 21  Crm Cd 2        float64
 22  Crm Cd 3        float64
 23  Crm Cd 4        float64
 24  LOCATION        object 
 25  Cross Street    object 
 26  LAT             object 
 27  LON             object 
dty

## 5. Initial inspection

### 5.1 Duplicates on DR_NO

In [9]:
duplicati = df['DR_NO'].duplicated().sum()   # Create the variable 'duplicati' by applying the '.duplicated' function to the
                                             # 'DR_NO' column, which returns a boolean value when a value is repeated;
                                             # the '.sum()' function is then applied to sum these values, obtaining
                                             # the total number of duplicate values in the column

print(f"Duplicates on DR_NO: {duplicati}")    # Display an f-string containing text and the value of the 'duplicati' variable

Duplicates on DR_NO: 57809


### 5.2 Null values per column

In [10]:
nulli_pct = (df.isnull().sum() / len(df) * 100).round(2)  # Create the variable 'nulli_pct' to track the percentage of
                                                          # null values for each column of the DataFrame

nulli_pct = nulli_pct[nulli_pct > 0].sort_values(ascending=False)  # Filter the variable 'nulli_pct' keeping values greater than 0
                                                                   # then sort the values in descending order

print("Percentage of null values per column:")  # Display the given string
print(nulli_pct)                                   # Display the 'nulli_pct' variable

Percentage of null values per column:
Crm Cd 4          99.99
Crm Cd 3          99.81
Crm Cd 2          93.27
Cross Street      83.71
Weapon Used Cd    66.73
Weapon Desc       66.73
Mocodes           12.15
Vict Sex          10.92
Vict Descent      10.92
Premis Desc        0.02
dtype: float64


### 5.3 Record distribution by year

In [11]:
df['anno_temp'] = pd.to_datetime(df['DATE OCC'], errors='coerce').dt.year
# Create the 'anno_temp' column by converting 'DATE OCC' to datetime format
# via pd.to_datetime(). 'errors='coerce'' turns values that cannot be
# converted into NaT instead of raising an error. '.dt.year' finally
# extracts only the year from each date.

print("Records per year:")  # Display the given string

print(df['anno_temp'].value_counts().sort_index())  # Display the count of occurrences for each year present
                                                    # in the 'anno_temp' column, sorted by index in ascending order
                                                    # (sort_index() uses ascending=True by default).

df.drop(columns='anno_temp', inplace=True)          # '.drop' removes the 'anno_temp' column from the DataFrame
                                                    # 'inplace=True' performs the operation on the DataFrame without creating a copy

/var/folders/wl/_8d4j_cs57qgchs9ckhqzkyr0000gn/T/ipykernel_8071/1731472415.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['anno_temp'] = pd.to_datetime(df['DATE OCC'], errors='coerce').dt.year


Records per year:
anno_temp
2010    209325
2011    200912
2012    201835
2013    192875
2014    195879
2015    168076
2016    283798
2017    231751
2018    229768
2019    218918
2020    199847
2021    209876
2022    235259
2023    232345
2024    127567
Name: count, dtype: int64


## 6. LAT/LON normalization

### Issue detected

The initial inspection revealed that the `LAT` and `LON` columns are of type `object` instead of `float64`. Investigating the two datasets separately revealed the cause:

- **2010–2019 dataset**: uses a **comma** as the decimal separator (e.g. `33,9825`) — European format
- **2020–2024 dataset**: uses a **period** as the decimal separator (e.g. `34.2124`) — standard Anglo-Saxon format

Pandas therefore infers different types: `float64` for the second dataset, `object` (string) for the first, because it doesn't recognize `33,9825` as a valid number.

### Failed solution attempt

An initial attempt with `pd.to_numeric(df['LAT'], errors='coerce')` produced **2,131,480 null values** (~68% of the dataset). `coerce` turns everything that isn't convertible into NaN, and since `33,9825` is not a valid number for Python, **all coordinates in the first dataset were destroyed**.

### Correct solution

Before converting, we normalize the format by replacing the comma with a period. The sequence becomes:

1. Force the type to string with `astype(str)` (also safe for the second dataset's floats, which become strings with a period)
2. Replace `,` with `.` via `str.replace`
3. Convert to `float` with `pd.to_numeric(errors='coerce')`

This way both formats are normalized and no valid data is lost.

In [12]:
df['LAT'] = df['LAT'].astype(str).str.replace(',', '.', regex=False)
df['LON'] = df['LON'].astype(str).str.replace(',', '.', regex=False)

df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')

print("Types:", df[['LAT', 'LON']].dtypes.to_dict())
print(f"Null LAT: {df['LAT'].isnull().sum()}")
print(f"Null LON: {df['LON'].isnull().sum()}")

Types: {'LAT': dtype('float64'), 'LON': dtype('float64')}
Null LAT: 0
Null LON: 0


In [13]:
print(f"Range LAT: {df['LAT'].min():.4f} → {df['LAT'].max():.4f}")
print(f"Range LON: {df['LON'].min():.4f} → {df['LON'].max():.4f}")
print(f"\nValues at (0, 0): {((df['LAT'] == 0) & (df['LON'] == 0)).sum()}")

Range LAT: 0.0000 → 34.7907
Range LON: -118.8279 → 0.0000

Values at (0, 0): 3148


In [14]:
#df.to_parquet('../../data/processed/crimes_merged.parquet')
print("Saved to data/processed/crimes_merged.parquet")

Saved to data/processed/crimes_merged.parquet


## 7. Conclusions — Plan Phase

## Conclusions — Plan Phase

**Dataset size**: 3,138,031 rows × 28 columns (~670 MB).

**Duplicates**: 57,809 duplicate records on DR_NO (~1.8%).
Will be inspected in EDA to define the handling strategy.

**Relevant null values**:
- `Crm Cd 2/3/4`: >93% null → dropped (one primary crime is sufficient)
- `Cross Street`: 84% null → dropped (redundant info with LOCATION + LAT/LON)
- `Weapon Used Cd / Weapon Desc`: 67% null → semantically "no weapon", kept
- `Mocodes`: 12% null → kept for modus operandi pattern analysis
- `Vict Sex / Vict Descent`: 11% null → recoded as "Unknown"

**Temporal anomalies identified**:
- Years 2015–2016 show anomalous values (a drop in 2015, a spike in 2016),
  likely related to the transition of the LAPD classification system
  (from UCR to NIBRS). To be verified and documented in the final report.
- Year 2024 incomplete (127,567 records): the truncation month needs to be verified
  in EDA and handled accordingly in the temporal analyses.

**Data types to convert**:
- `Date Rptd`, `DATE OCC` → datetime
- `TIME OCC` → formatted time
- `LAT`, `LON` → float (currently object, to be investigated in cleaning)
- `Vict Age` → handle sentinel values (0, negative)

**Operational decisions**:
- Reference period: 2010–2024 (full dataset)
- Full cleaning, not minimal, to support future analysis questions
- Feature engineering postponed to a dedicated notebook (03_feature_engineering)
- Possible sampling (99% confidence level / 1% margin of error) evaluated later if needed